# MDD Simple Models

This notebook adapts the SEED simple-model training structure for the MDD dataset. MDD uses 20 EEG channels, 200 time points per 1-second window, and 2 balanced classes. It keeps validation-based model selection and writes test predictions in the fixed DataLoader order.


## 1) Set data paths and inspect data shape

Use the official MDD train, validation, and test H5 files. The test set is read without labels and must keep its original order.

In [1]:
import os
import random
from dataclasses import dataclass

import h5py
import numpy as np

DATA_NAME = "MDD"
INDEX_PATH_TRAIN = f"G:/MLproject/course_project/{DATA_NAME}/train.h5"
INDEX_PATH_VAL = f"G:/MLproject/course_project/{DATA_NAME}/val.h5"
INDEX_PATH_TEST = f"G:/MLproject/course_project/{DATA_NAME}/test_x_only.h5"

with h5py.File(INDEX_PATH_TRAIN, "r") as f:
    print("keys:", list(f.keys()))
    print("x dtype:", f["X"].dtype)
    print("x shape:", f["X"].shape)
    print("y dtype:", f["y"].dtype)
    print("y shape:", f["y"].shape)
    print("unique labels:", np.unique(f["y"][()]))

keys: ['X', 'y']
x dtype: float32
x shape: (960, 20, 200)
y dtype: int64
y shape: (960,)
unique labels: [0 1]


## 2) Simple feature routes

The sklearn routes are intentionally small: bandpower, statistical summaries, covariance/correlation structure, and frequency-domain SVC baselines.

In [2]:
DEFAULT_BANDS = (
    ("delta", 1.0, 4.0),
    ("theta", 4.0, 8.0),
    ("alpha", 8.0, 13.0),
    ("beta", 13.0, 30.0),
    ("gamma", 30.0, 75.0),
)


def read_h5_xy(path):
    with h5py.File(path, "r") as f:
        return f["X"][()].astype(np.float32), f["y"][()].astype(np.int64)


def read_h5_x(path):
    with h5py.File(path, "r") as f:
        return f["X"][()].astype(np.float32)


@dataclass
class FeatureStandardizer:
    mean_: np.ndarray | None = None
    scale_: np.ndarray | None = None

    def fit(self, x):
        self.mean_ = x.mean(axis=0, keepdims=True)
        self.scale_ = x.std(axis=0, keepdims=True)
        self.scale_ = np.where(self.scale_ < 1e-12, 1.0, self.scale_)
        return self

    def transform(self, x):
        if self.mean_ is None or self.scale_ is None:
            raise RuntimeError("FeatureStandardizer must be fitted before transform().")
        return (x - self.mean_) / self.scale_

    def fit_transform(self, x):
        return self.fit(x).transform(x)


def make_band_power_matrix(x, sfreq=200.0, bands=DEFAULT_BANDS, eps=1e-12):
    x = np.asarray(x, dtype=np.float64)
    if x.ndim != 3:
        raise ValueError(f"Expected x shape (samples, channels, time), got {x.shape}.")
    n_times = x.shape[-1]
    x = x - x.mean(axis=-1, keepdims=True)
    window = np.hanning(n_times)
    freqs = np.fft.rfftfreq(n_times, d=1.0 / sfreq)
    df = freqs[1] - freqs[0] if len(freqs) > 1 else sfreq
    psd = (np.abs(np.fft.rfft(x * window, axis=-1)) ** 2) / (sfreq * np.sum(window**2))
    features = []
    for _, low, high in bands:
        mask = (freqs >= low) & (freqs < high)
        power = psd[..., mask].sum(axis=-1) * df
        features.append(np.maximum(power, eps))
    return np.stack(features, axis=-1)


def make_bandpower_features(x, sfreq=200.0, bands=DEFAULT_BANDS, log_power=True, relative=True, eps=1e-12):
    band_power = make_band_power_matrix(x, sfreq=sfreq, bands=bands, eps=eps)
    if relative:
        total_power = np.maximum(band_power.sum(axis=-1, keepdims=True), eps)
        band_power = band_power / total_power
    if log_power:
        band_power = np.log(band_power + eps)
    return band_power.reshape(band_power.shape[0], -1).astype(np.float32)


def make_differential_entropy_features(x, sfreq=200.0, bands=DEFAULT_BANDS, eps=1e-12):
    band_power = make_band_power_matrix(x, sfreq=sfreq, bands=bands, eps=eps)
    de = 0.5 * np.log(2.0 * np.pi * np.e * np.maximum(band_power, eps))
    return de.reshape(de.shape[0], -1).astype(np.float32)


def make_bandpower_de_features(x, sfreq=200.0, bands=DEFAULT_BANDS):
    bandpower = make_bandpower_features(x, sfreq=sfreq, bands=bands, log_power=True, relative=True)
    de = make_differential_entropy_features(x, sfreq=sfreq, bands=bands)
    return np.concatenate([bandpower, de], axis=1).astype(np.float32)


def make_statistical_features(x):
    x = np.asarray(x, dtype=np.float64)
    features = [
        x.mean(axis=-1),
        x.std(axis=-1),
        np.sqrt(np.mean(x**2, axis=-1)),
        x.min(axis=-1),
        x.max(axis=-1),
        np.ptp(x, axis=-1),
        np.quantile(x, 0.25, axis=-1),
        np.quantile(x, 0.50, axis=-1),
        np.quantile(x, 0.75, axis=-1),
    ]
    return np.concatenate(features, axis=1).astype(np.float32)


def make_covariance_features(x, eps=1e-6):
    x = np.asarray(x, dtype=np.float64)
    tri = np.triu_indices(x.shape[1])
    features = []
    for sample in x:
        centered = sample - sample.mean(axis=1, keepdims=True)
        cov = centered @ centered.T / max(centered.shape[1] - 1, 1)
        diag_scale = np.sqrt(np.maximum(np.diag(cov), eps))
        corr_like = cov / (diag_scale[:, None] * diag_scale[None, :])
        features.append(corr_like[tri])
    return np.asarray(features, dtype=np.float32)


def accuracy(y_true, y_pred):
    return float(np.mean(np.asarray(y_true) == np.asarray(y_pred)))


def predict_proba_aligned(model, x, classes=np.arange(3)):
    if hasattr(model, "predict_proba"):
        raw = model.predict_proba(x)
    else:
        raw = np.eye(len(classes), dtype=np.float64)[model.predict(x)]
    aligned = np.zeros((x.shape[0], len(classes)), dtype=np.float64)
    for src_idx, cls in enumerate(getattr(model, "classes_", classes)):
        dst_idx = int(np.where(classes == cls)[0][0])
        aligned[:, dst_idx] = raw[:, src_idx]
    return aligned

## 2b) DMD residual rank ensemble

This route learns one low-rank DMD transition system per class and per configuration, scores validation/test trials by class-conditional one-step residuals, then rank-normalizes scores before averaging across configurations.

In [3]:
@dataclass(frozen=True)
class DMDConfig:
    rank: int
    time_window: tuple[int, int]
    normalization: str = "channel_zscore"
    residual_metric: str = "mse"


def ensure_eeg_channel_time(x, expected_channels=None, expected_timepoints=None):
    expected_channels = CHANNELS if expected_channels is None else expected_channels
    expected_timepoints = TIME_POINTS if expected_timepoints is None else expected_timepoints
    x = np.asarray(x, dtype=np.float64)
    if x.ndim == 2:
        x = x[None, ...]
    if x.ndim != 3:
        raise ValueError(f"Expected EEG array with 3 dims, got shape {x.shape}.")
    if x.shape[1] == expected_channels:
        return x
    if x.shape[2] == expected_channels:
        return np.transpose(x, (0, 2, 1))
    if x.shape[1] == expected_timepoints and x.shape[2] != expected_timepoints:
        return np.transpose(x, (0, 2, 1))
    raise ValueError(f"Cannot infer channel/time axes for shape {x.shape}.")


def normalize_dmd_trials(x, normalization, eps=1e-8):
    x = np.asarray(x, dtype=np.float64)
    normalization = normalization.lower().replace("-", "_").replace(" ", "_")
    normalization = {
        "per_trial_z_score": "trial_zscore",
        "per_trial_zscore": "trial_zscore",
        "per_channel_z_score": "channel_zscore",
        "per_channel_zscore": "channel_zscore",
    }.get(normalization, normalization)
    if normalization == "raw":
        return x
    if normalization == "trial_zscore":
        mean = x.mean(axis=(1, 2), keepdims=True)
        scale = x.std(axis=(1, 2), keepdims=True)
    elif normalization == "channel_zscore":
        mean = x.mean(axis=2, keepdims=True)
        scale = x.std(axis=2, keepdims=True)
    else:
        raise ValueError(f"Unknown DMD normalization: {normalization}")
    return (x - mean) / np.maximum(scale, eps)


def select_dmd_window(x, config):
    start, end = config.time_window
    end = min(end, x.shape[-1])
    start = max(0, min(start, end - 2))
    if end - start < 2:
        raise ValueError(f"DMD time_window {config.time_window} is too short for shape {x.shape}.")
    return x[..., start:end]


def fit_dmd_system(X_train_class, config, svd_eps=1e-10):
    x = ensure_eeg_channel_time(X_train_class)
    x = select_dmd_window(x, config)
    x = normalize_dmd_trials(x, config.normalization)
    x_past = np.transpose(x[:, :, :-1], (1, 0, 2)).reshape(x.shape[1], -1)
    x_future = np.transpose(x[:, :, 1:], (1, 0, 2)).reshape(x.shape[1], -1)
    x_past = np.nan_to_num(x_past, copy=False)
    x_future = np.nan_to_num(x_future, copy=False)
    u, s, vt = np.linalg.svd(x_past, full_matrices=False)
    usable = int(np.sum(s > svd_eps))
    r = max(1, min(int(config.rank), usable, u.shape[1]))
    u_r = u[:, :r]
    s_r = s[:r]
    vt_r = vt[:r, :]
    return (x_future @ vt_r.T) @ ((1.0 / s_r)[:, None] * u_r.T)


def compute_dmd_residual(sample, A, config, eps=1e-8):
    residuals = compute_dmd_residual_batch(np.asarray(sample)[None, ...], A, config, eps=eps)
    return float(residuals[0])


def compute_dmd_residual_batch(X, A, config, eps=1e-8):
    metric = config.residual_metric.lower().replace("-", "_").replace(" ", "_")
    x = ensure_eeg_channel_time(X)
    x = select_dmd_window(x, config)
    x = normalize_dmd_trials(x, config.normalization)
    past = x[:, :, :-1]
    future = x[:, :, 1:]
    pred = np.einsum("ij,njt->nit", A, past, optimize=True)
    err = future - pred
    if metric == "mse":
        return np.mean(err**2, axis=(1, 2))
    if metric == "mae":
        return np.mean(np.abs(err), axis=(1, 2))
    if metric == "relative_error":
        num = np.sqrt(np.sum(err**2, axis=(1, 2)))
        den = np.sqrt(np.sum(future**2, axis=(1, 2)))
        return num / np.maximum(den, eps)
    if metric == "correlation_error":
        pred_flat = pred.reshape(pred.shape[0], -1)
        future_flat = future.reshape(future.shape[0], -1)
        pred_flat = pred_flat - pred_flat.mean(axis=1, keepdims=True)
        future_flat = future_flat - future_flat.mean(axis=1, keepdims=True)
        corr = np.sum(pred_flat * future_flat, axis=1) / np.maximum(
            np.linalg.norm(pred_flat, axis=1) * np.linalg.norm(future_flat, axis=1), eps
        )
        return 1.0 - corr
    raise ValueError(f"Unknown DMD residual metric: {config.residual_metric}")


def rank_transform_scores(scores):
    scores = np.asarray(scores, dtype=np.float64)
    ranks = np.empty_like(scores, dtype=np.float64)
    n_samples = scores.shape[0]
    if n_samples <= 1:
        return np.ones_like(scores, dtype=np.float64)
    for class_idx in range(scores.shape[1]):
        order = np.argsort(scores[:, class_idx], kind="mergesort")
        rank = np.empty(n_samples, dtype=np.float64)
        rank[order] = np.arange(n_samples, dtype=np.float64)
        ranks[:, class_idx] = rank / (n_samples - 1)
    return ranks


def build_default_dmd_configs():
    ranks = (2, 4, 6, 8, 10)
    windows = ((0, 400), (80, 320), (100, 300), (240, 400))
    normalizations = ("channel_zscore",)
    metrics = ("mse", "relative_error")
    return [
        DMDConfig(rank=rank, time_window=window, normalization=norm, residual_metric=metric)
        for rank in ranks
        for window in windows
        for norm in normalizations
        for metric in metrics
    ]


class DMDEnsembleClassifier:
    def __init__(self, configs=None, classes=None):
        self.configs = list(configs) if configs is not None else build_default_dmd_configs()
        self.classes = None if classes is None else np.asarray(classes, dtype=np.int64)
        self.systems_ = []

    def fit(self, X_train, y_train):
        X_train = ensure_eeg_channel_time(X_train)
        y_train = np.asarray(y_train, dtype=np.int64)
        self.classes_ = np.unique(y_train) if self.classes is None else self.classes
        self.systems_ = []
        for config in self.configs:
            class_systems = []
            for cls in self.classes_:
                class_x = X_train[y_train == cls]
                if class_x.shape[0] == 0:
                    raise ValueError(f"No training samples for class {cls}.")
                class_systems.append(fit_dmd_system(class_x, config))
            self.systems_.append(class_systems)
        return self

    def predict_scores(self, X):
        if not self.systems_:
            raise RuntimeError("DMDEnsembleClassifier must be fitted before predict_scores().")
        X = ensure_eeg_channel_time(X)
        config_rank_scores = []
        for config, class_systems in zip(self.configs, self.systems_):
            raw_scores = np.column_stack([
                -compute_dmd_residual_batch(X, A, config)
                for A in class_systems
            ])
            config_rank_scores.append(rank_transform_scores(raw_scores))
        return np.mean(config_rank_scores, axis=0)

    def predict(self, X, force_balanced=False):
        scores = self.predict_scores(X)
        if not force_balanced:
            return self.classes_[np.argmax(scores, axis=1)]
        n_samples, n_classes = scores.shape
        base = n_samples // n_classes
        counts = np.full(n_classes, base, dtype=int)
        counts[: n_samples % n_classes] += 1
        pred = np.full(n_samples, -1, dtype=int)
        margins = scores.max(axis=1) - np.partition(scores, -2, axis=1)[:, -2]
        for sample_idx in np.argsort(-margins):
            for class_idx in np.argsort(-scores[sample_idx]):
                if counts[class_idx] > 0:
                    pred[sample_idx] = self.classes_[class_idx]
                    counts[class_idx] -= 1
                    break
        return pred


def softmax_from_scores(scores):
    scores = np.asarray(scores, dtype=np.float64)
    shifted = scores - scores.max(axis=1, keepdims=True)
    exp_scores = np.exp(shifted)
    return exp_scores / np.maximum(exp_scores.sum(axis=1, keepdims=True), 1e-12)


def print_classification_report(name, y_true, y_pred):
    labels = np.arange(CLASSES)
    macro_f1 = f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    recalls = recall_score(y_true, y_pred, labels=labels, average=None, zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    ratio = np.bincount(np.asarray(y_pred, dtype=int), minlength=CLASSES) / len(y_pred)
    print(f"{name} validation accuracy: {accuracy(y_true, y_pred):.4f}")
    print(f"{name} macro F1: {macro_f1:.4f}")
    for cls, recall in zip(labels, recalls):
        print(f"{name} recall class {cls}: {recall:.4f}")
    print(f"{name} confusion matrix:\n{cm}")
    print(f"{name} predicted class ratio: {np.round(ratio, 4)}")


def fit_dmd_route(name="DMD Residual Rank Ensemble"):
    print("\n" + "=" * 60)
    print(f"Training {name}")
    classifier = DMDEnsembleClassifier(classes=np.arange(CLASSES))
    print(f"DMD configs: {len(classifier.configs)}")
    classifier.fit(train_x, train_y)
    train_scores = classifier.predict_scores(train_x)
    val_scores = classifier.predict_scores(val_x)
    test_scores = classifier.predict_scores(test_x)
    train_prob = softmax_from_scores(train_scores)
    val_prob = softmax_from_scores(val_scores)
    test_prob = softmax_from_scores(test_scores)
    train_pred = np.argmax(train_prob, axis=1)
    val_pred = np.argmax(val_prob, axis=1)
    train_acc = accuracy(train_y, train_pred)
    val_acc = accuracy(val_y, val_pred)
    print_classification_report(name, val_y, val_pred)
    print(f"{name} Train Accuracy: {train_acc:.4f}")
    print(f"{name} Val Accuracy: {val_acc:.4f}")
    return {"name": name, "train_acc": train_acc, "val_acc": val_acc, "val_prob": val_prob, "test_prob": test_prob}


## 3) Neural routes: EEGGRU, Simple MoE, Hybrid Router, Uncertainty-Aware Router

The notebook keeps the known strong `EEGGRU / BiGRU` configuration and adds three compact mixture-style neural baselines that can be trained with the same loop.

In [4]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from RNN import EEGGRU
from course_project.TEST_DATASET import TrainDataset, TestDataset

SEED = 3407
CHANNELS = 20
TIME_POINTS = 200
CLASSES = 2
SFREQ = 200.0
BATCH_SIZE = 64
EPOCHS = 120
PATIENCE = 18
MIN_DELTA = 1e-4
ENSEMBLE_MIN_VAL_ACC = 0.55
ENSEMBLE_WEIGHT_POWER = 2.0
RUN_DMD = False
RUN_TUNING_SWEEP = False
TOP_K_ENSEMBLE = 4
USE_CUDA = torch.cuda.is_available()
device = torch.device("cuda" if USE_CUDA else "cpu")
print("device:", device)


def set_global_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_global_seed(SEED)


def eeg_stat_features(x, eps=1e-6):
    mean = x.mean(dim=-1)
    std = x.std(dim=-1, unbiased=False)
    rms = torch.sqrt(torch.mean(x.square(), dim=-1) + eps)
    peak_to_peak = x.amax(dim=-1) - x.amin(dim=-1)
    return torch.cat([mean, std, rms, peak_to_peak], dim=1)


class SimpleMoEClassifier(nn.Module):
    def __init__(self, chans=CHANNELS, num_classes=CLASSES, hidden_dim=48, num_experts=4, dropout=0.25):
        super().__init__()
        kernels = (3, 7, 15, 31)[:num_experts]
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(chans, hidden_dim, kernel_size=kernel, padding=kernel // 2, bias=False),
                nn.BatchNorm1d(hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Conv1d(hidden_dim, hidden_dim, kernel_size=3, padding=1, groups=hidden_dim, bias=False),
                nn.BatchNorm1d(hidden_dim),
                nn.GELU(),
                nn.AdaptiveAvgPool1d(1),
                nn.Flatten(),
                nn.Linear(hidden_dim, num_classes),
            )
            for kernel in kernels
        ])
        self.router = nn.Sequential(
            nn.Linear(chans * 4, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, len(self.experts)),
        )

    def forward(self, x):
        route_logits = self.router(eeg_stat_features(x))
        route_weights = torch.softmax(route_logits, dim=1)
        expert_logits = torch.stack([expert(x) for expert in self.experts], dim=1)
        return torch.sum(route_weights.unsqueeze(-1) * expert_logits, dim=1)

    def clip_gradients(self, max_norm=1.0):
        torch.nn.utils.clip_grad_norm_(self.parameters(), max_norm)



class UncertaintyAwareHybridRouter(nn.Module):
    def __init__(self, chans=CHANNELS, num_classes=CLASSES, hidden_dim=64, dropout=0.30, uncertainty_threshold=0.18, uncertainty_sharpness=12.0):
        super().__init__()
        self.uncertainty_threshold = uncertainty_threshold
        self.uncertainty_sharpness = uncertainty_sharpness
        self.temporal_encoder = nn.Sequential(
            nn.Conv1d(chans, hidden_dim, kernel_size=9, padding=4, bias=False),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=5, padding=2, bias=False),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
        )
        self.base_head = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(hidden_dim, num_classes),
        )
        self.slow_expert = nn.Sequential(
            nn.Conv1d(chans, hidden_dim, kernel_size=33, padding=16, bias=False),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(hidden_dim, num_classes),
        )
        self.fast_expert = nn.Sequential(
            nn.Conv1d(chans, hidden_dim, kernel_size=5, padding=2, bias=False),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.AdaptiveMaxPool1d(1),
            nn.Flatten(),
            nn.Linear(hidden_dim, num_classes),
        )
        self.stat_expert = nn.Sequential(
            nn.Linear(chans * 4, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )
        self.router = nn.Sequential(
            nn.Linear(chans * 4 + hidden_dim + 3, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 3),
        )

    def uncertainty_features(self, base_logits):
        base_prob = torch.softmax(base_logits, dim=1)
        top2 = torch.topk(base_prob, k=2, dim=1).values
        confidence = top2[:, :1]
        margin = top2[:, :1] - top2[:, 1:2]
        entropy = -(base_prob * torch.log(base_prob.clamp_min(1e-8))).sum(dim=1, keepdim=True) / np.log(base_prob.shape[1])
        return torch.cat([confidence, margin, entropy], dim=1), margin

    def forward(self, x):
        stats = eeg_stat_features(x)
        encoded = self.temporal_encoder(x)
        pooled = encoded.mean(dim=-1)
        base_logits = self.base_head(encoded)
        uncertainty_features, margin = self.uncertainty_features(base_logits)
        route_input = torch.cat([stats, pooled, uncertainty_features], dim=1)
        route_weights = torch.softmax(self.router(route_input), dim=1)
        expert_logits = torch.stack([
            self.slow_expert(x),
            self.fast_expert(x),
            self.stat_expert(stats),
        ], dim=1)
        expert_mix = torch.sum(route_weights.unsqueeze(-1) * expert_logits, dim=1)
        uncertainty_weight = torch.sigmoid((self.uncertainty_threshold - margin) * self.uncertainty_sharpness)
        return (1.0 - uncertainty_weight) * base_logits + uncertainty_weight * expert_mix

    def clip_gradients(self, max_norm=1.0):
        torch.nn.utils.clip_grad_norm_(self.parameters(), max_norm)


class HybridExpertRouter(nn.Module):
    def __init__(self, chans=CHANNELS, num_classes=CLASSES, hidden_dim=64, dropout=0.30):
        super().__init__()
        self.temporal_encoder = nn.Sequential(
            nn.Conv1d(chans, hidden_dim, kernel_size=9, padding=4, bias=False),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=5, padding=2, bias=False),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
        )
        self.slow_expert = nn.Sequential(
            nn.Conv1d(chans, hidden_dim, kernel_size=33, padding=16, bias=False),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(hidden_dim, num_classes),
        )
        self.fast_expert = nn.Sequential(
            nn.Conv1d(chans, hidden_dim, kernel_size=5, padding=2, bias=False),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.AdaptiveMaxPool1d(1),
            nn.Flatten(),
            nn.Linear(hidden_dim, num_classes),
        )
        self.stat_expert = nn.Sequential(
            nn.Linear(chans * 4, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )
        self.router = nn.Sequential(
            nn.Linear(chans * 4 + hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 3),
        )

    def forward(self, x):
        stats = eeg_stat_features(x)
        encoded = self.temporal_encoder(x)
        pooled = encoded.mean(dim=-1)
        route_weights = torch.softmax(self.router(torch.cat([stats, pooled], dim=1)), dim=1)
        expert_logits = torch.stack([
            self.slow_expert(x),
            self.fast_expert(x),
            self.stat_expert(stats),
        ], dim=1)
        return torch.sum(route_weights.unsqueeze(-1) * expert_logits, dim=1)

    def clip_gradients(self, max_norm=1.0):
        torch.nn.utils.clip_grad_norm_(self.parameters(), max_norm)



device: cuda


## 6) Prepare DataLoader and train/validate

Train every route, record validation accuracy, and prepare probability outputs for final selection.

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score, recall_score
from sklearn.decomposition import PCA
from sklearn.svm import SVC

train_x, train_y = read_h5_xy(INDEX_PATH_TRAIN)
val_x, val_y = read_h5_xy(INDEX_PATH_VAL)
test_x = read_h5_x(INDEX_PATH_TEST)

train_ds = TrainDataset(INDEX_PATH_TRAIN)
val_ds = TrainDataset(INDEX_PATH_VAL)
test_ds = TestDataset(INDEX_PATH_TEST)

train_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, generator=train_generator, pin_memory=USE_CUDA)
train_eval_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=USE_CUDA)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=USE_CUDA)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, pin_memory=USE_CUDA)

print("train X:", train_x.shape, "train y:", train_y.shape)
print("val X:", val_x.shape, "val y:", val_y.shape)
print("test X:", test_x.shape)

def write_labels(path, labels):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for label in labels:
            f.write(f"{int(label)}\n")


def prediction_path_for_checkpoint(checkpoint_path):
    root, _ = os.path.splitext(checkpoint_path)
    return root + ".txt"


def write_labels_for_checkpoint(checkpoint_path, test_prob):
    checkpoint_pred_path = prediction_path_for_checkpoint(checkpoint_path)
    checkpoint_labels = np.argmax(test_prob, axis=1).astype(int).tolist()
    assert len(checkpoint_labels) == len(test_ds), f"Prediction count {len(checkpoint_labels)} != test sample count {len(test_ds)}"
    write_labels(checkpoint_pred_path, checkpoint_labels)
    print(f"Wrote checkpoint predictions to {checkpoint_pred_path}")
    return checkpoint_pred_path


def fit_tabular_route(name, feature_fn, model):
    print("\n" + "=" * 60)
    print(f"Training {name}")
    train_features = feature_fn(train_x)
    val_features = feature_fn(val_x)
    test_features = feature_fn(test_x)
    standardizer = FeatureStandardizer()
    train_features = standardizer.fit_transform(train_features)
    val_features = standardizer.transform(val_features)
    test_features = standardizer.transform(test_features)
    model.fit(train_features, train_y)
    train_prob = predict_proba_aligned(model, train_features, classes=np.arange(CLASSES))
    val_prob = predict_proba_aligned(model, val_features, classes=np.arange(CLASSES))
    test_prob = predict_proba_aligned(model, test_features, classes=np.arange(CLASSES))
    train_acc = accuracy(train_y, np.argmax(train_prob, axis=1))
    val_acc = accuracy(val_y, np.argmax(val_prob, axis=1))
    print("feature shape:", train_features.shape)
    print(f"{name} Train Accuracy: {train_acc:.4f}")
    print(f"{name} Val Accuracy: {val_acc:.4f}")
    return {"name": name, "train_acc": train_acc, "val_acc": val_acc, "val_prob": val_prob, "test_prob": test_prob}


def fit_reduced_tabular_route(name, feature_fn, reducer, model):
    print("\n" + "=" * 60)
    print(f"Training {name}")
    train_features = feature_fn(train_x)
    val_features = feature_fn(val_x)
    test_features = feature_fn(test_x)
    standardizer = FeatureStandardizer()
    train_features = standardizer.fit_transform(train_features)
    val_features = standardizer.transform(val_features)
    test_features = standardizer.transform(test_features)
    original_shape = train_features.shape
    train_features = reducer.fit_transform(train_features)
    val_features = reducer.transform(val_features)
    test_features = reducer.transform(test_features)
    model.fit(train_features, train_y)
    train_prob = predict_proba_aligned(model, train_features, classes=np.arange(CLASSES))
    val_prob = predict_proba_aligned(model, val_features, classes=np.arange(CLASSES))
    test_prob = predict_proba_aligned(model, test_features, classes=np.arange(CLASSES))
    train_acc = accuracy(train_y, np.argmax(train_prob, axis=1))
    val_acc = accuracy(val_y, np.argmax(val_prob, axis=1))
    print("feature shape:", original_shape, "->", train_features.shape)
    print(f"{name} Train Accuracy: {train_acc:.4f}")
    print(f"{name} Val Accuracy: {val_acc:.4f}")
    return {"name": name, "train_acc": train_acc, "val_acc": val_acc, "val_prob": val_prob, "test_prob": test_prob}


def train_torch_route(name, build_model, best_model_path, epochs=EPOCHS, lr=6e-4, weight_decay=8e-4, label_smoothing=0.04, scheduler_patience=6):
    print("\n" + "=" * 60)
    print(f"Training {name}")
    set_global_seed(SEED)
    model = build_model().to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=scheduler_patience, min_lr=1e-5)
    best_val_acc = -1.0
    best_epoch = 0
    bad_epochs = 0
    os.makedirs(os.path.dirname(best_model_path), exist_ok=True)
    for epoch in range(epochs):
        model.train()
        train_loss_sum = train_correct = train_num = 0
        for data, label in train_loader:
            data = data.to(device, dtype=torch.float32, non_blocking=USE_CUDA)
            label = label.to(device, dtype=torch.long, non_blocking=USE_CUDA)
            optimizer.zero_grad(set_to_none=True)
            output = model(data)
            loss = criterion(output, label)
            loss.backward()
            model.clip_gradients() if hasattr(model, "clip_gradients") else torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            batch_size = label.size(0)
            train_loss_sum += loss.item() * batch_size
            train_correct += (torch.argmax(output.detach(), dim=1) == label).sum().item()
            train_num += batch_size
        model.eval()
        val_loss_sum = val_correct = val_num = 0
        with torch.no_grad():
            for val_data, val_label in val_loader:
                val_data = val_data.to(device, dtype=torch.float32, non_blocking=USE_CUDA)
                val_label = val_label.to(device, dtype=torch.long, non_blocking=USE_CUDA)
                val_output = model(val_data)
                val_loss = criterion(val_output, val_label)
                batch_size = val_label.size(0)
                val_loss_sum += val_loss.item() * batch_size
                val_correct += (torch.argmax(val_output, dim=1) == val_label).sum().item()
                val_num += batch_size
        epoch_train_loss = train_loss_sum / train_num
        epoch_train_acc = train_correct / train_num
        epoch_val_loss = val_loss_sum / val_num
        epoch_val_acc = val_correct / val_num
        scheduler.step(epoch_val_acc)
        if epoch_val_acc > best_val_acc + MIN_DELTA:
            best_val_acc = epoch_val_acc
            best_epoch = epoch + 1
            bad_epochs = 0
            torch.save(model.state_dict(), best_model_path)
        else:
            bad_epochs += 1
        print(f"Epoch [{epoch + 1:03d}/{epochs}] | Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.4f} | Best: {best_val_acc:.4f} @ {best_epoch:03d}")
        if bad_epochs >= PATIENCE:
            print(f"Early stopping at epoch {epoch + 1}; best epoch was {best_epoch}.")
            break
    model.load_state_dict(torch.load(best_model_path, map_location=device))
    model.eval()

    def collect_prob(loader):
        probs = []
        with torch.no_grad():
            for batch in loader:
                data = batch[0] if isinstance(batch, (tuple, list)) else batch
                data = data.to(device, dtype=torch.float32, non_blocking=USE_CUDA)
                probs.append(torch.softmax(model(data), dim=1).cpu().numpy())
        return np.concatenate(probs, axis=0)

    train_prob = collect_prob(train_eval_loader)
    val_prob = collect_prob(val_loader)
    test_prob = collect_prob(test_loader)
    checkpoint_pred_path = write_labels_for_checkpoint(best_model_path, test_prob)
    train_acc = accuracy(train_y, np.argmax(train_prob, axis=1))
    val_acc = accuracy(val_y, np.argmax(val_prob, axis=1))
    print(f"{name} Best Val Accuracy: {best_val_acc:.4f} (epoch {best_epoch})")
    print(f"{name} Recomputed Train Accuracy: {train_acc:.4f}")
    print(f"{name} Recomputed Val Accuracy: {val_acc:.4f}")
    return {"name": name, "train_acc": train_acc, "val_acc": val_acc, "val_prob": val_prob, "test_prob": test_prob, "prediction_path": checkpoint_pred_path}


model_results = []
model_results.append(fit_reduced_tabular_route("BandpowerDE + PCA + LogisticRegression", lambda x: make_bandpower_de_features(x, sfreq=SFREQ), PCA(n_components=0.98, random_state=SEED), LogisticRegression(C=0.5, class_weight="balanced", max_iter=1000, random_state=SEED, solver="lbfgs")))
model_results.append(fit_tabular_route("Covariance + LogisticRegression", make_covariance_features, LogisticRegression(C=0.15, class_weight="balanced", max_iter=1000, random_state=SEED, solver="lbfgs")))
model_results.append(fit_tabular_route("Bandpower + SVC", lambda x: make_bandpower_features(x, sfreq=SFREQ), SVC(C=0.8, kernel="rbf", gamma="scale", class_weight="balanced", probability=True, random_state=SEED)))
model_results.append(fit_tabular_route("Statistical + RandomForest", make_statistical_features, RandomForestClassifier(n_estimators=300, max_depth=8, class_weight="balanced", random_state=SEED, n_jobs=-1)))
if RUN_DMD:
    model_results.append(fit_dmd_route("DMD Residual Rank Ensemble"))
else:
    print("\nSkipping DMD Residual Rank Ensemble because RUN_DMD = False.")

torch_route_specs = [
    # Keep the strongest latest single model and search nearby capacity/dropout settings.
    ("Simple MoE", lambda: SimpleMoEClassifier(chans=CHANNELS, num_classes=CLASSES, hidden_dim=48, num_experts=4, dropout=0.25), f"course_project/{DATA_NAME}/best_mdd_simple_models_moe.pt", EPOCHS, 7e-4, 8e-4, 0.04, 6),
    ("Simple MoE h64 dp30", lambda: SimpleMoEClassifier(chans=CHANNELS, num_classes=CLASSES, hidden_dim=64, num_experts=4, dropout=0.30), f"course_project/{DATA_NAME}/best_mdd_simple_models_moe_h64_dp30.pt", EPOCHS, 6e-4, 1e-3, 0.05, 6),
    ("Simple MoE h64 dp35", lambda: SimpleMoEClassifier(chans=CHANNELS, num_classes=CLASSES, hidden_dim=64, num_experts=4, dropout=0.35), f"course_project/{DATA_NAME}/best_mdd_simple_models_moe_h64_dp35.pt", EPOCHS, 5e-4, 1e-3, 0.06, 7),
    ("Simple MoE h80 dp35", lambda: SimpleMoEClassifier(chans=CHANNELS, num_classes=CLASSES, hidden_dim=80, num_experts=4, dropout=0.35), f"course_project/{DATA_NAME}/best_mdd_simple_models_moe_h80_dp35.pt", EPOCHS, 4e-4, 1.2e-3, 0.06, 7),
    ("Simple MoE h96 dp40", lambda: SimpleMoEClassifier(chans=CHANNELS, num_classes=CLASSES, hidden_dim=96, num_experts=4, dropout=0.40), f"course_project/{DATA_NAME}/best_mdd_simple_models_moe_h96_dp40.pt", EPOCHS, 4e-4, 1.5e-3, 0.07, 8),
    ("Hybrid Router h48 dp30", lambda: HybridExpertRouter(chans=CHANNELS, num_classes=CLASSES, hidden_dim=48, dropout=0.30), f"course_project/{DATA_NAME}/best_mdd_simple_models_hybrid_h48_dp30.pt", EPOCHS, 6e-4, 1e-3, 0.05, 6),
    ("EEGGRU / BiGRU", lambda: EEGGRU(chans=CHANNELS, hidden_dim=96, num_layers=2, num_classes=CLASSES, dropout=0.35, bidirectional=True), f"course_project/{DATA_NAME}/best_mdd_simple_models_bigru.pt", EPOCHS, 6e-4, 8e-4, 0.04, 6),
    ("EEGGRU h128 dp45", lambda: EEGGRU(chans=CHANNELS, hidden_dim=128, num_layers=2, num_classes=CLASSES, dropout=0.45, bidirectional=True), f"course_project/{DATA_NAME}/best_mdd_simple_models_bigru_h128_dp45.pt", EPOCHS, 4e-4, 1.2e-3, 0.06, 7),
    ("Uncertainty Router h64 t18", lambda: UncertaintyAwareHybridRouter(chans=CHANNELS, num_classes=CLASSES, hidden_dim=64, dropout=0.30, uncertainty_threshold=0.18, uncertainty_sharpness=12.0), f"course_project/{DATA_NAME}/best_mdd_simple_models_uncertainty_h64_t18.pt", EPOCHS, 6e-4, 1e-3, 0.05, 6),
]

if RUN_TUNING_SWEEP:
    torch_route_specs.extend([
        ("Uncertainty Router h96 t15", lambda: UncertaintyAwareHybridRouter(chans=CHANNELS, num_classes=CLASSES, hidden_dim=96, dropout=0.35, uncertainty_threshold=0.15, uncertainty_sharpness=14.0), f"course_project/{DATA_NAME}/best_mdd_simple_models_uncertainty_h96_t15.pt", EPOCHS, 4e-4, 1.2e-3, 0.06, 7),
        ("Uncertainty Router h64 t22", lambda: UncertaintyAwareHybridRouter(chans=CHANNELS, num_classes=CLASSES, hidden_dim=64, dropout=0.25, uncertainty_threshold=0.22, uncertainty_sharpness=10.0), f"course_project/{DATA_NAME}/best_mdd_simple_models_uncertainty_h64_t22.pt", EPOCHS, 6e-4, 8e-4, 0.04, 6),
        ("Hybrid Router h96 dp40", lambda: HybridExpertRouter(chans=CHANNELS, num_classes=CLASSES, hidden_dim=96, dropout=0.40), f"course_project/{DATA_NAME}/best_mdd_simple_models_hybrid_h96_dp40.pt", EPOCHS, 4e-4, 1.2e-3, 0.06, 7),
    ])

for route_name, build_model, checkpoint_path, epochs, lr, weight_decay, label_smoothing, scheduler_patience in torch_route_specs:
    model_results.append(train_torch_route(route_name, build_model, checkpoint_path, epochs=epochs, lr=lr, weight_decay=weight_decay, label_smoothing=label_smoothing, scheduler_patience=scheduler_patience))

train X: (960, 20, 200) train y: (960,)
val X: (640, 20, 200) val y: (640,)
test X: (800, 20, 200)

Training BandpowerDE + PCA + LogisticRegression
feature shape: (960, 200) -> (960, 77)
BandpowerDE + PCA + LogisticRegression Train Accuracy: 0.9458
BandpowerDE + PCA + LogisticRegression Val Accuracy: 0.8719

Training Covariance + LogisticRegression
feature shape: (960, 210)
Covariance + LogisticRegression Train Accuracy: 0.9417
Covariance + LogisticRegression Val Accuracy: 0.8328

Training Bandpower + SVC
feature shape: (960, 100)
Bandpower + SVC Train Accuracy: 0.9313
Bandpower + SVC Val Accuracy: 0.8688

Training Statistical + RandomForest
feature shape: (960, 180)
Statistical + RandomForest Train Accuracy: 0.9844
Statistical + RandomForest Val Accuracy: 0.8781

Training DMD Residual Rank Ensemble
DMD configs: 40
DMD Residual Rank Ensemble validation accuracy: 0.7578
DMD Residual Rank Ensemble macro F1: 0.7528
DMD Residual Rank Ensemble recall class 0: 0.6156
DMD Residual Rank Ensemb

## 7) Select final method and write MDD.txt

Compare the best single model with validation-weighted soft voting. Write one integer label per test sample in `shuffle=False` order.

In [6]:
weights = np.array([max(result["val_acc"] - ENSEMBLE_MIN_VAL_ACC, 0.0) ** ENSEMBLE_WEIGHT_POWER for result in model_results], dtype=np.float64)
top_mask = np.ones(len(model_results), dtype=bool)
if TOP_K_ENSEMBLE is not None and TOP_K_ENSEMBLE > 0 and len(model_results) > TOP_K_ENSEMBLE:
    top_indices = np.argsort([result["val_acc"] for result in model_results])[-TOP_K_ENSEMBLE:]
    top_mask = np.zeros(len(model_results), dtype=bool)
    top_mask[top_indices] = True
    weights = np.where(top_mask, weights, 0.0)
if np.allclose(weights.sum(), 0.0):
    weights = top_mask.astype(np.float64) / max(top_mask.sum(), 1)
else:
    weights = weights / weights.sum()

weighted_val_prob = sum(weight * result["val_prob"] for weight, result in zip(weights, model_results))
weighted_test_prob = sum(weight * result["test_prob"] for weight, result in zip(weights, model_results))
ensemble_val_pred = np.argmax(weighted_val_prob, axis=1)
ensemble_test_pred = np.argmax(weighted_test_prob, axis=1)
ensemble_val_acc = accuracy(val_y, ensemble_val_pred)

best_single_idx = int(np.argmax([result["val_acc"] for result in model_results]))
best_single = model_results[best_single_idx]
best_single_val_acc = best_single["val_acc"]

print("\nModel validation summary")
for result, weight in zip(model_results, weights):
    print(f"{result['name']}: train_acc={result['train_acc']:.4f}, val_acc={result['val_acc']:.4f}, ensemble_weight={weight:.4f}")
print(f"Ensemble Val Accuracy: {ensemble_val_acc:.4f}")
print(f"Best Single Model: {best_single['name']} ({best_single_val_acc:.4f})")

if ensemble_val_acc >= best_single_val_acc:
    selected_method_name = "Validation-weighted soft voting ensemble"
    selected_val_acc = ensemble_val_acc
    selected_test_prob = weighted_test_prob
else:
    selected_method_name = best_single["name"]
    selected_val_acc = best_single_val_acc
    selected_test_prob = best_single["test_prob"]

selected_test_pred = np.argmax(selected_test_prob, axis=1)
all_test_labels = selected_test_pred.astype(int).tolist()
assert len(all_test_labels) == len(test_ds), f"Prediction count {len(all_test_labels)} != test sample count {len(test_ds)}"



output_path = f"course_project/{DATA_NAME}/{DATA_NAME}.txt"
write_labels(output_path, all_test_labels)
print(f"Final selected method: {selected_method_name} | Val Accuracy: {selected_val_acc:.4f}")
print(f"Wrote {len(all_test_labels)} predictions to {output_path}")


Model validation summary
BandpowerDE + PCA + LogisticRegression: train_acc=0.9458, val_acc=0.8719, ensemble_weight=0.0000
Covariance + LogisticRegression: train_acc=0.9417, val_acc=0.8328, ensemble_weight=0.0000
Bandpower + SVC: train_acc=0.9313, val_acc=0.8688, ensemble_weight=0.0000
Statistical + RandomForest: train_acc=0.9844, val_acc=0.8781, ensemble_weight=0.0000
DMD Residual Rank Ensemble: train_acc=0.7760, val_acc=0.7578, ensemble_weight=0.0000
Simple MoE: train_acc=0.9688, val_acc=0.9500, ensemble_weight=0.2534
Simple MoE h64 dp30: train_acc=0.9615, val_acc=0.9531, ensemble_weight=0.2574
Simple MoE h64 dp35: train_acc=0.9698, val_acc=0.9484, ensemble_weight=0.2514
Simple MoE h80 dp35: train_acc=0.9656, val_acc=0.9359, ensemble_weight=0.0000
Simple MoE h96 dp40: train_acc=0.9604, val_acc=0.9375, ensemble_weight=0.2378
Hybrid Router h48 dp30: train_acc=0.9948, val_acc=0.9141, ensemble_weight=0.0000
EEGGRU / BiGRU: train_acc=0.9885, val_acc=0.8672, ensemble_weight=0.0000
EEGGRU h